# Chapter 19 — Alignment

**Book alignment:** Embeddings From First Principles, Chapter 19

**Question this notebook isolates:** Try the most constrained map that meets the target.
Does a nonlinear MLP beat a closed-form linear map on preservation — or (the committed Wave
3 bake-off) does it reconstruct *worse* than Procrustes while costing the longest fit and
carrying seed variance? And always run the **null map** `T(x)=x` first.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. Orthogonal Procrustes vs ridge — closed forms, in NumPy

In [ ]:
n, d = 300, 40
X = rng.standard_normal((n, d))
X /= np.linalg.norm(X, axis=1, keepdims=True)
Rtrue, _ = np.linalg.qr(rng.standard_normal((d, d)))
Y = X @ Rtrue + rng.standard_normal((n, d)) * 0.05    # same shape, different pose

# Procrustes: R = U V^T from the SVD of X^T Y - preserves all within-X distances/angles
U, _, Vt = np.linalg.svd(X.T @ Y)
R = U @ Vt
# ridge: any linear map (rotation + scale + shear)
W = np.linalg.solve(X.T @ X + 1e-2 * np.eye(d), X.T @ Y)

def recon(P):
    Z = X @ P; Z /= np.linalg.norm(Z, axis=1, keepdims=True)
    Yn = Y / np.linalg.norm(Y, axis=1, keepdims=True)
    return float(np.mean(np.sum(Z * Yn, 1)))

print(f"null map T(x)=x : recon {recon(np.eye(d)):.3f}")
print(f"Procrustes      : recon {recon(R):.3f}   (only re-orients)")
print(f"ridge           : recon {recon(W):.3f}")
assert recon(R) > recon(np.eye(d))
# Procrustes preserves the Gram matrix of X exactly:
assert np.allclose((X @ R) @ (X @ R).T, X @ X.T, atol=1e-6)
print("Procrustes keeps every within-A inner product; ridge does not")

## 2. The bake-off on RELATE (Wave 3): the MLP does not win

In [ ]:
nl = art("wave3", "nonlinear-vs-linear-unpaired")["pairs"]["bge-large vs mxbai-large"]
for rung in ("procrustes", "ridge", "mlp"):
    v = nl[rung]
    print(f"  {rung:11} recon {v['coordinate_reconstruction']:.3f}  10-NN {v['neighborhood_at10']:.3f}"
          f"  fit {v['fit_seconds']:.2f}s  seed-std {v['seed_std_reconstruction']:.4f}")

# Procrustes keeps the most neighbourhood structure; the MLP reconstructs WORSE than
# Procrustes, costs the longest fit, and its result moves with the seed
assert nl["procrustes"]["neighborhood_at10"] >= max(nl["ridge"]["neighborhood_at10"],
                                                    nl["mlp"]["neighborhood_at10"])
assert nl["mlp"]["seed_std_reconstruction"] > 0.0
assert nl["ridge"]["seed_std_reconstruction"] == 0.0
assert nl["mlp"]["coordinate_reconstruction"] < nl["procrustes"]["coordinate_reconstruction"]
assert nl["mlp"]["fit_seconds"] > nl["ridge"]["fit_seconds"]
print("\nexpressiveness bought nothing a closed-form linear map did not already have")

## 3. Reconstruction is not preservation

In [ ]:
rungs = art("wave3", "ladder-8property-matrix")["pairs"]["mpnet-base vs bge-large"]["rungs"]
r_ridge, r_proc = rungs["ridge"], rungs["procrustes"]
print(f"ridge      recon {r_ridge['coordinate_reconstruction']:.2f}  neighbourhood {r_ridge['neighborhood_at10']:.2f}")
print(f"procrustes recon {r_proc['coordinate_reconstruction']:.2f}  neighbourhood {r_proc['neighborhood_at10']:.2f}")
assert r_ridge["coordinate_reconstruction"] > r_proc["coordinate_reconstruction"]   # ridge fits coords better
assert r_proc["neighborhood_at10"] > r_ridge["neighborhood_at10"]                   # procrustes keeps neighbours better
print("cosine-to-target and 'neighbours stay neighbours' come apart - optimise and report the second")

## What we earned

The alignment family runs null ⊂ Procrustes ⊂ linear ⊂ CCA ⊂ nonlinear, ordered by
overfitting risk. Run the null map first. On RELATE the MLP led on nothing — it reconstructed
worse than Procrustes, took the longest to fit, and its result moved with the seed.
Coordinate reconstruction (cosine to target) and semantic preservation (neighbours,
rankings) diverge routinely; optimise and evaluate for the second. Rule: **the most
constrained map that meets the preservation target.**

**Notebook 20 / Chapter 20** assembles alignment into a scoped artifact — the bridge — with
an explicit `usable_for` list.